In [ ]:
import os, glob, numpy as np, pandas as pd, psycopg, pyarrow.parquet as pq
from dotenv import load_dotenv
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

part = sorted(glob.glob("../data/embeddings/part-*.parquet"))[0]
t = pq.read_table(part)
ids = t.column("recommendationid").to_numpy()
vecs = t.column("embedding").combine_chunks().flatten().to_numpy().reshape(-1, 384)
norms = np.linalg.norm(vecs, axis=1)
print(part, vecs.shape, f"norms: min {norms.min():.4f}, max {norms.max():.4f}")   # both should be ~1.0000

In [ ]:
texts = pd.read_sql("SELECT recommendationid, review FROM reviews WHERE recommendationid = ANY(%(ids)s)",
                    conn, params={"ids": ids.tolist()}).set_index("recommendationid").loc[ids, "review"]

In [ ]:
crash = [i for i, r in enumerate(texts) if "crash" in r.lower() and 40 < len(r) < 300]
q = crash[0]
sims = vecs @ vecs[q]                       # vectors are unit length, so dot product = cosine similarity
top = np.argsort(-sims)[1:6]                # [0] is the review itself
print(f"QUERY: {texts.iloc[q][:200]!r}\n")
for j in top:
    print(f"{sims[j]:.2f}  {texts.iloc[j][:200]!r}")
print(f"\nrandom pairs, for comparison: mean similarity {np.mean(vecs[np.random.default_rng(0).integers(0, len(vecs), 1000)] @ vecs[q]):.2f}")